# PACE Phase 9 — Full-budget GA objective ablations (analysis only)

This notebook trains the **original 4-member AMP recognition ensemble**, **all six original 300-tree MIC models**, and the **300-tree HC50 model** from the exact same curated 90D descriptors and labels. It then re-runs five objective variants × two random seeds, preserving 240 population size, 45 generations, 24 elites and the original mutation/crossover operators. It uses the same design budget for every arm and reports all objectives under the same common model suite. 

**No manuscript editing occurs.** The notebook checks source SHA-256 hashes and stops if inputs are different. Upload the existing `PACE_Phase8_ESM2_results.zip` from your previous Colab run. It may consume substantial CPU/GPU time depending on the runtime. A freshly refitted model is not guaranteed to reproduce an earlier pickled model bit-for-bit.

In [ ]:
!pip -q install scikit-learn biopython requests numpy

In [ ]:
from google.colab import files
from pathlib import Path
print('Upload the previously generated PACE_Phase8_ESM2_results.zip')
received=files.upload()
assert 'PACE_Phase8_ESM2_results.zip' in received, 'Upload exactly PACE_Phase8_ESM2_results.zip'
Path('/content/PACE_Phase8_ESM2_results.zip').write_bytes(received['PACE_Phase8_ESM2_results.zip'])
print('Uploaded archive:',len(received['PACE_Phase8_ESM2_results.zip']),'bytes')

In [ ]:
# Script is embedded from the package so all original source checks and analyses are reproducible
from pathlib import Path
SOURCE = '"""PACE Phase 9 — Fully refitted GA ablations from verified Phase 8 source output.\nNo manuscript or original author file edits. Quantitative MIC/HC50 labels and 90D features\ncome from user\'s ESM2 Phase 8 archive, and original FASTA labels from fixed-source GitHub.\n"""\nimport io, csv, json, time, zipfile, hashlib, collections, sys, importlib.util\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.ensemble import (ExtraTreesClassifier,RandomForestClassifier,\n    HistGradientBoostingClassifier,ExtraTreesRegressor)\nfrom sklearn.svm import SVC\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nimport requests\nROOT=Path(\'/content\');OUT=ROOT/\'pace_phase9_colab\';OUT.mkdir(exist_ok=True)\nZ=ROOT/\'PACE_Phase8_ESM2_results.zip\'\nassert Z.exists(), f\'Missing {Z}; upload Phase8 ESM2 results ZIP first\'\nwith zipfile.ZipFile(Z) as z:\n    manifest=json.loads(z.read(\'manifest.json\'))\n    db=np.load(io.BytesIO(z.read(\'all_embeddings_and_features.npz\')))\n    seq_ref=np.asarray(db[\'sequences\']);X_ref=np.asarray(db[\'X90\'])\n    predtab=z.read(\'quantitative_predictions.csv\').decode()\n    (OUT/\'features_original_source.py\').write_bytes(z.read(\'pace_original_features.py\'))\nnpz_dict={s:i for i,s in enumerate(seq_ref)}\nsha=hashlib.sha256()\nsource={}\nfor name,info in manifest[\'source_files\'].items():\n    print(\'Downloading\',name,flush=True)\n    r=requests.get(info[\'url\'],timeout=120);r.raise_for_status()\n    assert hashlib.sha256(r.content).hexdigest()==info[\'sha256\'],f\'Hash mismatch {name}\'\n    source[name]=r.content.decode(\'utf-8-sig\')\n(OUT/\'input_source_manifest.json\').write_text(json.dumps(manifest,indent=2))\nspec=importlib.util.spec_from_file_location(\'pace_features\',OUT/\'features_original_source.py\')\nfeatures=importlib.util.module_from_spec(spec);spec.loader.exec_module(features)\nALPHABET_CANON=set(\'ACDEFGHIKLMNPQRSTVWY\')\ndef parse_fasta(txt):\n    seqs=[];buf=[]\n    for ln in txt.splitlines():\n        if ln.startswith(\'>\'):\n            if buf:seqs.append(\'\'.join(buf));buf=[]\n        else:buf.append(ln.strip())\n    if buf:seqs.append(\'\'.join(buf))\n    return seqs\ndef clean_many(raw):\n    r=[]\n    for x in parse_fasta(raw):\n        s=features.clean(x)\n        if s is not None and 5<=len(s)<=60:r.append(s)\n    return r\ntrainPos=clean_many(source[\'AMP_train\']);trainNeg=clean_many(source[\'nonAMP_train\'])\ntestPos=clean_many(source[\'AMP_test\']);testNeg=clean_many(source[\'nonAMP_test\'])\nextPos=clean_many(source[\'external_AMP\']);extNeg=clean_many(source[\'external_nonAMP\'])\ndef dedup(pos,neg):\n    d={}\n    for s in pos:d.setdefault(s,1)\n    for s in neg:d.setdefault(s,0)\n    return d\ntrain=dedup(trainPos,trainNeg);test={s:y for s,y in dedup(testPos,testNeg).items() if s not in train}\nseen=set(train)|set(test)\nexternal={s:y for s,y in dedup(extPos,extNeg).items() if s not in seen}\nassert (len(train),sum(train.values()),len(test),sum(test.values()),len(external),sum(external.values()))==(6271,3124,1568,796,1848,671)\ncl={**train,**test}; ic=np.array([npz_dict[s] for s in cl]);yc=np.array(list(cl.values()))\nprint(\'Fitting original PACE four-member soft-voting classifier on 7839 train+heldout peptides\',flush=True)\ncls_models={\n    \'GB\':HistGradientBoostingClassifier(max_iter=400,learning_rate=.07,random_state=2024),\n    \'ET\':ExtraTreesClassifier(n_estimators=400,random_state=2024,n_jobs=4),\n    \'SVM\':Pipeline([(\'s\',StandardScaler()),(\'m\',SVC(C=4,gamma=\'scale\',probability=True,random_state=2024))]),\n    \'RF\':RandomForestClassifier(n_estimators=400,random_state=2024,n_jobs=4)\n}\nfor name,model in cls_models.items():\n    model.fit(X_ref[ic],yc);print(\'classifier fitted\',name,flush=True)\nqs=[r for r in csv.DictReader(io.StringIO(predtab)) if r[\'representation\']==\'90D_PACE\']\nquant={}\nfor r in qs:quant.setdefault(r[\'endpoint\'],{}).setdefault(r[\'sequence\'],float(r[\'observed_log10\']))\nexp={\'E. faecium\':780,\'S. aureus\':4128,\'K. pneumoniae\':743,\'A. baumannii\':306,\'P. aeruginosa\':2517,\'E. cloacae\':304,\'HC50\':1271}\nassert all(len(quant[x])==n for x,n in exp.items()),{x:len(quant.get(x,{})) for x in exp}\nreg={}\nfor target in list(exp)[:-1]:\n    q=quant[target];ix=np.array([npz_dict[s] for s in q]);y=np.array(list(q.values()))\n    m=ExtraTreesRegressor(n_estimators=300,random_state=2024,n_jobs=4);m.fit(X_ref[ix],y);reg[target]=m\n    print(\'MIC fitted\',target,len(y),flush=True)\nq=quant[\'HC50\'];ix=np.array([npz_dict[s] for s in q]);y=np.array(list(q.values()))\nhem=ExtraTreesRegressor(n_estimators=300,random_state=2024,n_jobs=4);hem.fit(X_ref[ix],y)\nprint(\'HC50 fitted\',len(y),flush=True)\nPOOL=sorted(set(s for s,y in {**train,**test,**external}.items() if y==1)|set().union(*(set(q.keys()) for q in quant.values())))\nK=3; inv=collections.defaultdict(list);sets=[]\nfor i,s in enumerate(POOL):\n    a=set(s[j:j+3] for j in range(len(s)-2));sets.append(a)\n    for k in a:inv[k].append(i)\ndef maximum_similarity(s):\n    a={s[j:j+3] for j in range(len(s)-2)};cnt=collections.Counter()\n    for k in a:\n        for i in inv.get(k,()):cnt[i]+=1\n    best=0.\n    for i,shared in cnt.items():\n        j=shared/(len(a)+len(sets[i])-shared)\n        if j>best:best=j\n    return best\ncache={};counts={\'features\':0}\ndef get_scores(seqs):\n    new=[s for s in dict.fromkeys(seqs) if s not in cache]\n    if new:\n        X=features.featurize_many(new)\n        p=np.mean([m.predict_proba(X)[:,1] for m in cls_models.values()],axis=0)\n        mm=np.column_stack([reg[t].predict(X) for t in reg]);hc=hem.predict(X)\n        for i,s in enumerate(new):\n            sim=maximum_similarity(s);mean=float(mm[i].mean());worst=float(mm[i].max());ti=float(hc[i]-mean);charge=features.net_charge(s)\n            cache[s]={\'p_amp\':float(p[i]),\'mic_mean\':mean,\'mic_worst\':worst,\'hc50\':float(hc[i]),\'ti\':ti,\'sim\':sim,\'charge\':float(charge),\'length\':len(s),\'sequence\':s}\n        counts[\'features\']+=len(new)\n    return [cache[s] for s in seqs]\nprovenance={\'source\':\'Official refit of PACE 90D models from Phase8 curated labels and pinned FASTA sources, reconstructed from original published code\',\n  \'quantitative_labels_source\':\'Phase8 quantitative_predictions 90D_PACE observed_log10 (uniquely deduplicated by sequence and endpoint)\',\n  \'classifier_training_n\':len(cl),\'classifier_names\':list(cls_models),\'classifier_trees\':400,\'regressor_trees\':300,\n  \'novelty_known_peptides\':len(POOL),\'ga_seeds\':[7,29],\'original_pace_source\':\'PACE_reproducibility_code.zip\',\n  \'limits\':\'Newly fitted models may not bitwise reproduce prior author weights; same descriptor code and model settings\',\n  \'phase8_sha256\':hashlib.sha256(Z.read_bytes()).hexdigest()}\n(OUT/\'full_protocol_model_provenance.json\').write_text(json.dumps(provenance,indent=2))\nALPHABET=list(\'ADEFGHIKLNPQRSTVWY\');LMIN=12;LMAX=30;POP=240;GENS=45;ELITE=24\nTARGETS=list(reg)\nprovenance={ \'scope\':\'Analysis-only reconstructed proxy models, not exact PACE original fitted model weights\',\n  \'original_classifier_retrained\':False,\'proxy_classifier_training\':\'3416 labeled prior heldout+external sequences; not suitable for claiming independent classification validation\',\n  \'proxy_amp_model\':\'ExtraTreesClassifier 100 trees\',\n  \'MIC_HC50_models\':\'original 90D, ExtraTrees 80 trees, full training set (original 300 trees)\',\n  \'novelty_pool\':\'all 12979 sequences represented in Colab embeddings (conservative differs from original original AMP+MIC+HC50 pool)\',\n  \'GA_settings\':{\'pop\':POP,\'generations\':GENS,\'elite\':ELITE,\'crossover\':.6,\'mutate_prob\':.9,\'seeds\':[7,29]},\n  \'fitness_original\':\'2*p_amp - mean_log10mic - 0.5*worst_log10mic + 0.6*clip(ti,-2,4) - 2*max(0,sim-0.55) - 0.35*max(0,abs(q-6)-3)\',\n  \'input_sha256\':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [Z,ROOT/\'PACE_reproducibility_code.zip\',ROOT/\'Pasted text(6).txt\',ROOT/\'Cleaned_hemolytic_data.csv\']}}\n(OUT/\'proxy_scope_and_checksums.json\').write_text(json.dumps(provenance,indent=2))\ncache={};counts={\'features\':0}\ndef get_scores(seqs):\n    fresh=[s for s in dict.fromkeys(seqs) if s not in cache]\n    if fresh:\n        xx=features.featurize_many(fresh)\n        p=clf.predict_proba(xx)[:,1]\n        mm=np.column_stack([reg[t].predict(xx) for t in TARGETS]);hc=hem.predict(xx)\n        for i,s in enumerate(fresh):\n            sim=maximum_similarity(s)\n            mean=float(np.mean(mm[i]));worst=float(np.max(mm[i]));ti=float(hc[i]-mean);q=features.net_charge(s)\n            cache[s]={\'p_amp\':float(p[i]),\'mic_mean\':mean,\'mic_worst\':worst,\'hc50\':float(hc[i]),\'ti\':ti,\'sim\':sim,\'charge\':float(q),\'length\':len(s),\'sequence\':s}\n        counts[\'features\']+=len(fresh)\n    return [cache[s] for s in seqs]\n\ndef fitness(v,arm):\n    a=2*v[\'p_amp\']-v[\'mic_mean\'];b=.5*v[\'mic_worst\'];c=.6*float(np.clip(v[\'ti\'],-2,4));d=2*max(0,v[\'sim\']-.55);e=.35*max(0,abs(v[\'charge\']-6)-3)\n    if arm==\'full\':return a-b+c-d-e\n    if arm==\'no_hemolysis\':return a-b-d-e\n    if arm==\'no_novelty\':return a-b+c-e\n    if arm==\'no_worst_case\':return a+c-d-e\n    if arm==\'potency_recognition_only\':return a-e\n    raise KeyError(arm)\n\ndef random_seq(rng):return \'\'.join(rng.choice(ALPHABET,int(rng.integers(LMIN,LMAX+1))))\ndef mutate(s,rng):\n    s=list(s);op=rng.random()\n    if op<.70 or len(s)<=LMIN:\n        for _ in range(int(rng.integers(1,4))):s[int(rng.integers(len(s)))]=str(rng.choice(ALPHABET))\n    elif op<.85 and len(s)<LMAX:s.insert(int(rng.integers(len(s)+1)),str(rng.choice(ALPHABET)))\n    else:del s[int(rng.integers(len(s)))]\n    return \'\'.join(s)\ndef crossover(a,b,rng):\n    s=a[:int(rng.integers(1,len(a)))]+b[int(rng.integers(1,len(b))):]\n    if len(s)>LMAX:s=s[:LMAX]\n    while len(s)<LMIN:s+=str(rng.choice(ALPHABET))\n    return s\n\ndef pair_sim(a,b):\n    u=set(a[i:i+3] for i in range(len(a)-2));v=set(b[i:i+3] for i in range(len(b)-2))\n    return len(u&v)/len(u|v)\n\ndef select_leads(unique,arm):\n    order=sorted(unique,key=lambda s:fitness(cache[s],arm),reverse=True)\n    passed=[];rejected_by=collections.Counter()\n    for s in order:\n        v=cache[s]\n        if v[\'p_amp\']<.9:rejected_by[\'p_amp\']+=1;continue\n        if v[\'mic_worst\']>np.log10(32):rejected_by[\'mic\']+=1;continue\n        if v[\'ti\']<1:rejected_by[\'selectivity\']+=1;continue\n        if v[\'sim\']>=.55:rejected_by[\'novelty\']+=1;continue\n        passed.append(s)\n    leads=[]\n    for s in passed:\n        if all(pair_sim(s,t)<=.45 for t in leads):leads.append(s)\n        if len(leads)==20:break\n    return leads,len(passed),dict(rejected_by)\n\ndef run(arm,seed):\n    rng=np.random.default_rng(seed)\n    pop=[random_seq(rng) for _ in range(POP)];seen=set();history=[]\n    for g in range(1,GENS+1):\n        rec=get_scores(pop);seen.update(pop)\n        fs=[fitness(s,arm) for s in rec]\n        order=np.argsort(-np.array(fs)); pop=[pop[j] for j in order];sc=[rec[j] for j in order];fs=[fs[j] for j in order]\n        history.append({\'arm\':arm,\'seed\':seed,\'gen\':g,\'best_native_fitness\':fs[0],\'population_mean_p_amp\':float(np.mean([s[\'p_amp\'] for s in sc])),\n          \'population_mean_mic_log10\':float(np.mean([s[\'mic_mean\'] for s in sc])),\'population_mean_ti_log10\':float(np.mean([s[\'ti\'] for s in sc])),\n          \'mean_length\':float(np.mean([s[\'length\'] for s in sc])),\'mean_novelty_similarity\':float(np.mean([s[\'sim\'] for s in sc]))})\n        if g==GENS:break\n        nxt=pop[:ELITE]\n        while len(nxt)<POP:\n            def tournament():\n                ids=rng.integers(0,len(pop),3)\n                return pop[int(max(ids,key=lambda i:fs[i]))]\n            c=crossover(tournament(),tournament(),rng) if rng.random()<.6 else tournament()\n            if rng.random()<.9:c=mutate(c,rng)\n            if LMIN<=len(c)<=LMAX:nxt.append(c)\n        pop=nxt\n    leads,feasible,rej=select_leads(seen,arm)\n    values=[cache[s] for s in leads]\n    out={\'arm\':arm,\'seed\':seed,\'unique_scored\':len(seen),\'feasible_hard_filters\':feasible,\'n_diverse_leads\':len(leads),\n         \'best_native_fitness\':max(fitness(cache[s],arm) for s in seen),\n         \'fraction_known_sim_above_0_55\':np.mean([cache[s][\'sim\']>.55 for s in seen]),\n         \'fraction_length_29_30_all\':np.mean([cache[s][\'length\']>=29 for s in seen]),\n         \'lead_avg_length\':float(np.mean([v[\'length\'] for v in values])) if values else None,\n         \'lead_avg_mic_log10\':float(np.mean([v[\'mic_mean\'] for v in values])) if values else None,\n         \'lead_avg_worst_mic_log10\':float(np.mean([v[\'mic_worst\'] for v in values])) if values else None,\n         \'lead_avg_ti_log10\':float(np.mean([v[\'ti\'] for v in values])) if values else None,\n         \'lead_avg_p_amp\':float(np.mean([v[\'p_amp\'] for v in values])) if values else None,\n         \'lead_avg_known_sim\':float(np.mean([v[\'sim\'] for v in values])) if values else None,\n         \'lead_avg_full_fitness\':float(np.mean([fitness(v,\'full\') for v in values])) if values else None,\n         \'filter_reject_reasons\':rej}\n    print(\'RUN\',json.dumps({k:v for k,v in out.items() if k!=\'filter_reject_reasons\'}),flush=True)\n    return out,history,[{\'arm\':arm,\'seed\':seed,\'rank\':i+1,**cache[s], \'full_fitness\':fitness(cache[s],\'full\'), \'native_fitness\':fitness(cache[s],arm)} for i,s in enumerate(leads)],seen\n\ndef save_table(path,rows):\n    if not rows:return\n    cols=list(rows[0]);\n    with path.open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,cols,extrasaction=\'ignore\');w.writeheader();w.writerows(rows)\n\nif __name__==\'__main__\':\n    stats=[];hist=[];leads=[];runsequences={};start=time.time()\n    arms=[\'full\',\'no_hemolysis\',\'no_novelty\',\'no_worst_case\',\'potency_recognition_only\']\n    for seed in [7,29]:\n        for arm in arms:\n            st,h,l,seen=run(arm,seed);stats.append(st);hist.extend(h);leads.extend(l);runsequences[(arm,seed)]=seen\n        print(\'NO_NOVELTY_TRAJECTORY_OVERLAP\',seed,len(runsequences[(\'full\',seed)]&runsequences[(\'no_novelty\',seed)]),flush=True)\n    save_table(OUT/\'phase9_ablation_run_summary.csv\',stats)\n    save_table(OUT/\'phase9_ablation_history.csv\',hist)\n    save_table(OUT/\'phase9_ablation_leads.csv\',leads)\n    report={\'runtime_seconds\':time.time()-start,\'model_type\':\'Original PACE classifier family GB/ET/SVM/RF and 300-tree MIC+hemolysis ExtraTrees; refitted with original data + original descriptor engine\',\'arms\':arms,\'seeds\':[7,29],\n            \'runs\':stats,\'novelty_trajectories\':{str(seed):{\'full_union_n\':len(runsequences[(\'full\',seed)]),\'no_novelty_union_n\':len(runsequences[(\'no_novelty\',seed)]),\'symmetric_difference\':len(runsequences[(\'full\',seed)]^runsequences[(\'no_novelty\',seed)])} for seed in [7,29]},\'total_unique_predictions_across_runs\':len(cache)}\n    (OUT/\'phase9_ablation_summary.json\').write_text(json.dumps(report,indent=2))\n    from zipfile import ZipFile,ZIP_DEFLATED\n    zpath=ROOT/\'PACE_Phase9_Full_GA_Colab_Results.zip\'\n    with ZipFile(zpath,\'w\',ZIP_DEFLATED) as z:\n        for p in OUT.glob(\'*\'):\n            if p.is_file() and p.suffix not in {\'.pkl\',\'.npz\'}:z.write(p,arcname=p.name)\n    print(\'DONE\',zpath,flush=True)\n'
Path('/content/PACE_Phase9_Full_GA_Colab_Runner.py').write_text(SOURCE)
exec(compile(SOURCE,'PACE_Phase9_Full_GA_Colab_Runner.py','exec'),{'__name__':'__main__'})

In [ ]:
from google.colab import files
files.download('/content/PACE_Phase9_Full_GA_Colab_Results.zip')